In [24]:
%load_ext autoreload
%autoreload 2
 
import numpy as np
import pandas as pd
 
from assignment_1.modules.models import GapModel, GapRegressor, split_last, SECONDS_PER_DAY
from assignment_1.modules.util import load_csv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
DATA_DIR = "../data"  # notebook runs from notebooks/
OFFSET = 35 * SECONDS_PER_DAY
 
train = load_csv("task_6_train.csv", DATA_DIR)
test = load_csv("task_6_test.csv", DATA_DIR)

In [26]:
print(f"train {train.shape}, test {test.shape}, test users unique: {test.user.is_unique}")
print(f"test users in train: {test.user.isin(train.user).mean():.1%}")
hist_items = train.groupby("user")["item"].apply(set)
in_hist = [i in hist_items.get(u, set()) for u, i in zip(test.user, test.item)]
print(f"test item already in user's train history: {np.mean(in_hist):.1%}")

train (1301245, 4), test (55403, 3), test users unique: True
test users in train: 100.0%
test item already in user's train history: 0.0%


In [27]:
s = train.sort_values(["user", "timestamp"], kind="stable")
d = s.groupby("user")["timestamp"].diff().dropna()
print(f"{len(d):,} gaps; negative: {(d < 0).sum()}, zero: {(d == 0).mean():.1%}")
print(f"users with only 1 train row: {(train.groupby('user').size() == 1).mean():.1%}")
(d / DAY).describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1)

1,245,842 gaps; negative: 0, zero: 0.0%
users with only 1 train row: 0.0%


count    1245842.0
mean          70.9
std          878.6
min            0.0
10%            1.0
25%            4.0
50%           13.0
75%           46.0
90%          143.9
99%          829.0
max       720204.0
Name: timestamp, dtype: float64

In [28]:
tr, val = split_last(train)
y = val["target"].to_numpy()
mae = lambda p: float(np.mean(np.abs(np.asarray(p) - y)))
print(f"{len(val):,} validation users")

55,403 validation users


In [29]:
print(f"per-user median, all gaps, no shrinkage: {mae(GapModel(k=None, lam=0).fit(tr).predict(val)):,.0f}")

per-user median, all gaps, no shrinkage: 12,052,136


In [30]:
rows = [{"k": k or "all", "lam": lam, "mae": mae(GapModel(k=k, lam=lam).fit(tr).predict(val))}
        for k in [5, 10, 20, None] for lam in [0.5, 1, 1.5, 2, 3, 5]]
pd.DataFrame(rows).sort_values("mae").head(8)

,k,lam,mae
8,10,1.5,1.013722e+07
14,20,1.5,1.014591e+07
20,all,1.5,1.015260e+07
2,5,1.5,1.015390e+07
9,10,2.0,1.016555e+07
15,20,2.0,1.017258e+07
7,10,1.0,1.017410e+07
21,all,2.0,1.017904e+07


In [31]:
gm = GapModel(k=10, lam=1.5).fit(tr)
bound = gm.bound(val)
print(f"bound applies: {(bound > gm.predict(val)).mean():.1%}")
print(f"with bound: {mae(np.fmax(gm.predict(val), bound)):,.0f}   (expect 8,531,601)")

bound applies: 13.1%
with bound: 8,531,601   (expect 8,531,601)


In [32]:
extra = (y - bound)[bound > gm.predict(val)]
print(f"median extra delay past the bound: {np.nanmedian(extra) / SECONDS_PER_DAY:.1f} days")
print(f"shipped (k 10, λ 2, +35 days): {mae(GapModel(k=10, lam=2.0, bound_offset=OFFSET).fit(tr).predict(val)):,.0f}")

median extra delay past the bound: 35.0 days
shipped (k 10, λ 2, +35 days): 8,426,265


In [34]:
for lam in [1.5, 2.0]:
    m = GapRegressor(base=GapModel(k=10, lam=lam), bound_offset=OFFSET).fit(tr)
    print(f"λ={lam}: {mae(m.predict(val)):,.0f}")

λ=1.5: 8,012,169
λ=2.0: 8,019,054
